# Liquidation Strategy Inspection

This inspection walkthrough reviews the V1 liquidation strategy engine with the synthetic sample data. It uses the existing loaders and the existing liquidation strategy service.

The goal is to make the liquidation process easy to inspect: scenario inputs, cash-buffer preservation, eligibility, gross liquidation allocations, post-haircut cash raised, dilution, shortfall, and asset-group allocations.

## Imports And Setup

This section imports the shared setup helper, existing project loaders, domain labels, and liquidation strategy engine. It also defines small display helpers for readable tables. These helpers are local inspection aids and are not business logic that the application will call.

In [ ]:
from decimal import Decimal

import pandas as pd
from _notebook_setup import SAMPLE_DATA_DIR, configure_display

from lmt_calibration.domain import AssetGroup
from lmt_calibration.engines import StressedLiquidationPosition, calculate_liquidation_strategy
from lmt_calibration.loaders import (
    load_funds_csv,
    load_investor_classes_csv,
    load_liquidation_strategies_json,
    load_liquidity_stresses_csv,
    load_lmt_parameters_csv,
    load_market_stresses_csv,
    load_positions_csv,
    load_redemption_scenarios_csv,
    load_scenario_definitions_csv,
)

configure_display()


def money(value: Decimal | int | float | None) -> str:
    if value is None:
        return ""
    return f"{Decimal(value):,.2f}"


def rate(value: Decimal | int | float | None) -> str:
    if value is None:
        return ""
    return f"{Decimal(value) * Decimal('100'):.2f}%"


def days(value: int | None) -> str:
    if value is None:
        return ""
    return str(value)

## Sample Data Loaded

This section loads every sample dataset through the existing loaders. This matters because the inspection should use the same validated domain objects that the application code uses, rather than bypassing validation with ad hoc CSV reads.

Interpret the output as a quick inventory of the sample inputs available for the liquidation review.

In [3]:
funds = load_funds_csv(SAMPLE_DATA_DIR / "funds.csv")
positions = load_positions_csv(SAMPLE_DATA_DIR / "positions.csv")
investor_classes = load_investor_classes_csv(SAMPLE_DATA_DIR / "investor_classes.csv")
redemption_scenarios = load_redemption_scenarios_csv(SAMPLE_DATA_DIR / "redemption_scenarios.csv")
market_stresses = load_market_stresses_csv(SAMPLE_DATA_DIR / "market_stresses.csv")
liquidity_stresses = load_liquidity_stresses_csv(SAMPLE_DATA_DIR / "liquidity_stresses.csv")
scenario_definitions = load_scenario_definitions_csv(SAMPLE_DATA_DIR / "scenario_definitions.csv")
lmt_parameters = load_lmt_parameters_csv(SAMPLE_DATA_DIR / "lmt_parameters.csv")
liquidation_strategies = load_liquidation_strategies_json(
    SAMPLE_DATA_DIR / "liquidation_strategies.json"
)

pd.DataFrame(
    [
        {"dataset": "funds", "records": len(funds)},
        {"dataset": "positions", "records": len(positions)},
        {"dataset": "investor_classes", "records": len(investor_classes)},
        {"dataset": "redemption_scenarios", "records": len(redemption_scenarios)},
        {"dataset": "market_stresses", "records": len(market_stresses)},
        {"dataset": "liquidity_stresses", "records": len(liquidity_stresses)},
        {"dataset": "scenario_definitions", "records": len(scenario_definitions)},
        {"dataset": "lmt_parameters", "records": len(lmt_parameters)},
        {"dataset": "liquidation_strategies", "records": len(liquidation_strategies)},
    ]
)

,dataset,records
0,funds,1
1,positions,9
2,investor_classes,5
3,redemption_scenarios,3
4,market_stresses,3
5,liquidity_stresses,3
6,scenario_definitions,4
7,lmt_parameters,2
8,liquidation_strategies,4


## Scenario Overview

This section shows the scenario definitions selected for inspection. It matters because each scenario maps a fund, redemption assumption, liquidity stress, liquidation strategy, and LMT parameter set into one engine run.

Interpret this table as the review menu: each row is one representative liquidation strategy run.

In [4]:
fund_by_key = {(fund.fund_id, fund.as_of_date): fund for fund in funds}
redemption_by_id = {scenario.redemption_scenario_id: scenario for scenario in redemption_scenarios}
market_stress_by_id = {stress.market_stress_id: stress for stress in market_stresses}
liquidity_stress_by_id = {stress.liquidity_stress_id: stress for stress in liquidity_stresses}
strategy_by_id = {strategy.liquidation_strategy_id: strategy for strategy in liquidation_strategies}
parameters_by_key = {
    (parameters.fund_id, parameters.as_of_date, parameters.parameter_set_id): parameters
    for parameters in lmt_parameters
}

pd.DataFrame(
    [
        {
            "scenario_id": scenario.scenario_id,
            "fund_id": scenario.fund_id,
            "as_of_date": str(scenario.as_of_date),
            "redemption_scenario_id": scenario.redemption_scenario_id,
            "market_stress_id": scenario.market_stress_id,
            "liquidity_stress_id": scenario.liquidity_stress_id,
            "liquidation_strategy_id": scenario.liquidation_strategy_id,
            "lmt_parameter_set_id": scenario.lmt_parameter_set_id,
            "strategy_type": strategy_by_id[scenario.liquidation_strategy_id].strategy_type.value,
        }
        for scenario in scenario_definitions
    ]
)

,scenario_id,fund_id,as_of_date,redemption_scenario_id,market_stress_id,liquidity_stress_id,liquidation_strategy_id,lmt_parameter_set_id,strategy_type
0,moderate_redemption_most_liquid,lux_dynamic_allocation,2026-06-30,moderate_redemption_pressure,base_market_conditions,normal_liquidity_capacity,cash_then_liquid_assets,board_approved_base,most_liquid_first
1,platform_outflow_pro_rata,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,europe_equity_downturn,reduced_equity_capacity,portfolio_profile_pro_rata,board_approved_base,pro_rata
2,platform_outflow_hybrid,lux_dynamic_allocation,2026-06-30,severe_platform_outflow,europe_equity_downturn,reduced_equity_capacity,partial_cash_then_pro_rata,board_approved_base,hybrid
3,extreme_outflow_custom_weights,lux_dynamic_allocation,2026-06-30,extreme_broad_based_outflow,global_equity_selloff,severe_liquidity_squeeze,balanced_custom_weights,conservative_liquidity_buffer,custom_weights


## Local Inspection Helpers

The liquidation engine expects already-stressed liquidation positions and a redemption amount. This section prepares those inputs from the loaded sample data so the engine can be called.

This matters because the inspection reviews the liquidation strategy engine, rather than implementing a full asset-stress or liability-stress service. The helper uses documented sample assumptions for review only: non-stress liquidity scenarios use zero haircut, stressed liquidity scenarios use positive haircut rates from the loaded position assumptions and liquidity stress multiplier. Market stress identifiers are shown for traceability, while the liquidation inspection keeps market values at the sample position values because market-stress methodology is outside this inspection scope.

In [5]:
SELLABLE_GROUPS = {
    AssetGroup.REVERSE_REPO,
    AssetGroup.LISTED_ETF,
    AssetGroup.LISTED_EQUITY,
}


def scenario_investors(scenario):
    return [
        investor
        for investor in investor_classes
        if investor.fund_id == scenario.fund_id and investor.as_of_date == scenario.as_of_date
    ]


def redemption_rows(scenario):
    fund = fund_by_key[(scenario.fund_id, scenario.as_of_date)]
    redemption = redemption_by_id[scenario.redemption_scenario_id]
    rows = []
    for investor in scenario_investors(scenario):
        redemption_amount = (
            fund.nav
            * investor.nav_share_rate
            * investor.stress_redemption_rate
            * redemption.redemption_multiplier
        )
        rows.append(
            {
                "scenario_id": scenario.scenario_id,
                "client_class": investor.client_class.value,
                "nav_share_rate": investor.nav_share_rate,
                "stress_redemption_rate": investor.stress_redemption_rate,
                "redemption_multiplier": redemption.redemption_multiplier,
                "redemption_amount": redemption_amount,
            }
        )
    return rows


def total_redemption_amount(scenario):
    return sum((row["redemption_amount"] for row in redemption_rows(scenario)), Decimal("0"))


def stressed_haircut_rate(position, liquidity_stress):
    if position.asset_group is AssetGroup.CASH:
        return Decimal("0")
    if liquidity_stress.liquidity_stress_multiplier == Decimal("1.00"):
        return Decimal("0")
    return min(
        position.base_haircut_rate * liquidity_stress.liquidity_stress_multiplier, Decimal("1")
    )


def stressed_liquidity_capacity_rate(position, liquidity_stress):
    if position.asset_group is AssetGroup.CASH:
        return Decimal("1")
    return position.base_liquidity_capacity_rate / liquidity_stress.liquidity_stress_multiplier


def scenario_positions(scenario):
    liquidity_stress = liquidity_stress_by_id[scenario.liquidity_stress_id]
    return [
        StressedLiquidationPosition(
            position_id=position.position_id,
            asset_group=position.asset_group,
            stressed_market_value=position.market_value,
            stressed_haircut_rate=stressed_haircut_rate(position, liquidity_stress),
            stressed_liquidity_capacity_rate=stressed_liquidity_capacity_rate(
                position, liquidity_stress
            ),
            settlement_days=position.settlement_days,
            maturity_days=position.maturity_days,
            notional_amount=position.notional_amount,
        )
        for position in positions
        if position.fund_id == scenario.fund_id and position.as_of_date == scenario.as_of_date
    ]


def is_eligible(position, stress_horizon_days):
    if position.asset_group not in SELLABLE_GROUPS:
        return False
    if position.stressed_market_value is None or position.stressed_market_value <= Decimal("0"):
        return False
    if position.asset_group is AssetGroup.REVERSE_REPO:
        if position.maturity_days is None:
            return False
        return position.maturity_days + position.settlement_days <= stress_horizon_days
    return position.settlement_days <= stress_horizon_days


def run_scenario(scenario):
    fund = fund_by_key[(scenario.fund_id, scenario.as_of_date)]
    liquidity_stress = liquidity_stress_by_id[scenario.liquidity_stress_id]
    strategy = strategy_by_id[scenario.liquidation_strategy_id]
    parameters = parameters_by_key[
        (scenario.fund_id, scenario.as_of_date, scenario.lmt_parameter_set_id)
    ]
    redemption_amount = total_redemption_amount(scenario)
    liquidation_positions = scenario_positions(scenario)
    result = calculate_liquidation_strategy(
        scenario_id=scenario.scenario_id,
        fund=fund,
        positions=liquidation_positions,
        redemption_amount=redemption_amount,
        strategy=strategy,
        lmt_parameters=parameters,
        stress_horizon_days=liquidity_stress.stress_horizon_days,
    )
    return {
        "scenario": scenario,
        "fund": fund,
        "liquidity_stress": liquidity_stress,
        "market_stress": market_stress_by_id[scenario.market_stress_id],
        "strategy": strategy,
        "parameters": parameters,
        "redemption_amount": redemption_amount,
        "positions": liquidation_positions,
        "result": result,
    }


runs = [run_scenario(scenario) for scenario in scenario_definitions]

## Scenario Inputs

This section summarizes the key inputs passed into each liquidation run: NAV, total redemption amount, selected strategy, liquidity horizon, and configured LMT buffer.

Interpret `total_redemption_rate` as the liability-side pressure being tested. The liquidation engine then tries to raise this amount while respecting the strategy, haircuts, capacity, settlement, maturity, and cash-buffer rules.

In [6]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "market_stress_id": run["market_stress"].market_stress_id,
            "liquidity_stress_id": run["liquidity_stress"].liquidity_stress_id,
            "stress_horizon_days": run["liquidity_stress"].stress_horizon_days,
            "nav": money(run["fund"].nav),
            "total_redemption_amount": money(run["redemption_amount"]),
            "total_redemption_rate": rate(run["redemption_amount"] / run["fund"].nav),
            "minimum_buffer_rate": rate(run["parameters"].minimum_buffer_rate),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,market_stress_id,liquidity_stress_id,stress_horizon_days,nav,total_redemption_amount,total_redemption_rate,minimum_buffer_rate
0,moderate_redemption_most_liquid,most_liquid_first,base_market_conditions,normal_liquidity_capacity,5,"100,000,000.00","11,500,000.00",11.50%,5.00%
1,platform_outflow_pro_rata,pro_rata,europe_equity_downturn,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,5.00%
2,platform_outflow_hybrid,hybrid,europe_equity_downturn,reduced_equity_capacity,5,"100,000,000.00","17,250,000.00",17.25%,5.00%
3,extreme_outflow_custom_weights,custom_weights,global_equity_selloff,severe_liquidity_squeeze,3,"100,000,000.00","23,000,000.00",23.00%,8.00%


## Redemption Amount By Investor Class

This section shows how the sample investor-class assumptions contribute to each scenario's redemption amount. It matters because the liquidation engine receives a single redemption amount, but the review should still see where that amount came from.

Interpret larger class-level rows as the investor groups driving cash demand in the liquidation run.

In [7]:
redemption_table = pd.DataFrame(
    {
        **row,
        "nav_share_rate": rate(row["nav_share_rate"]),
        "stress_redemption_rate": rate(row["stress_redemption_rate"]),
        "redemption_multiplier": str(row["redemption_multiplier"]),
        "redemption_amount": money(row["redemption_amount"]),
    }
    for scenario in scenario_definitions
    for row in redemption_rows(scenario)
)
redemption_table

,scenario_id,client_class,nav_share_rate,stress_redemption_rate,redemption_multiplier,redemption_amount
0,moderate_redemption_most_liquid,retail,30.00%,8.00%,1.00,"2,400,000.00"
1,moderate_redemption_most_liquid,institutional,25.00%,12.00%,1.00,"3,000,000.00"
2,moderate_redemption_most_liquid,platform,25.00%,18.00%,1.00,"4,500,000.00"
3,moderate_redemption_most_liquid,fund_of_funds,15.00%,10.00%,1.00,"1,500,000.00"
4,moderate_redemption_most_liquid,seed_capital,5.00%,2.00%,1.00,"100,000.00"
5,platform_outflow_pro_rata,retail,30.00%,8.00%,1.50,"3,600,000.00"
6,platform_outflow_pro_rata,institutional,25.00%,12.00%,1.50,"4,500,000.00"
7,platform_outflow_pro_rata,platform,25.00%,18.00%,1.50,"6,750,000.00"
8,platform_outflow_pro_rata,fund_of_funds,15.00%,10.00%,1.50,"2,250,000.00"
9,platform_outflow_pro_rata,seed_capital,5.00%,2.00%,1.50,"150,000.00"


## Cash Buffer Calculation

This section shows how much cash is available above the configured minimum buffer. It matters because V1 strategies preserve the minimum buffer and should not automatically drain all cash.

Interpret `cash_above_buffer` as the maximum cash that a cash-using strategy can consume before selling eligible non-cash assets.

In [8]:
cash_buffer_rows = []
for run in runs:
    cash_total = sum(
        (
            position.stressed_market_value or Decimal("0")
            for position in run["positions"]
            if position.asset_group is AssetGroup.CASH
        ),
        Decimal("0"),
    )
    minimum_cash_buffer = run["fund"].nav * run["parameters"].minimum_buffer_rate
    cash_buffer_rows.append(
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "cash_total": money(cash_total),
            "minimum_cash_buffer": money(minimum_cash_buffer),
            "cash_above_buffer": money(max(cash_total - minimum_cash_buffer, Decimal("0"))),
            "cash_used": money(run["result"].cash_used),
            "minimum_cash_buffer_preserved": run["result"].minimum_cash_buffer_preserved,
        }
    )

pd.DataFrame(cash_buffer_rows)

,scenario_id,strategy_type,cash_total,minimum_cash_buffer,cash_above_buffer,cash_used,minimum_cash_buffer_preserved
0,moderate_redemption_most_liquid,most_liquid_first,"12,000,000.00","5,000,000.00","7,000,000.00","7,000,000.00",True
1,platform_outflow_pro_rata,pro_rata,"12,000,000.00","5,000,000.00","7,000,000.00",0.00,True
2,platform_outflow_hybrid,hybrid,"12,000,000.00","5,000,000.00","7,000,000.00","3,500,000.00",True
3,extreme_outflow_custom_weights,custom_weights,"12,000,000.00","8,000,000.00","4,000,000.00",0.00,True


## Eligible Assets

This section shows which positions can provide usable liquidity within each scenario's stress horizon. It matters because the engine excludes repo financing from ordinary liquidation and applies settlement-day and reverse-repo maturity rules before allocating sales.

Interpret `eligible_for_liquidation` as whether the asset can be sold or matured within the horizon for this one-period run. Cash appears here for context but is not treated as an ordinary sellable asset.

In [9]:
eligible_rows = []
for run in runs:
    horizon = run["liquidity_stress"].stress_horizon_days
    for position in run["positions"]:
        available_capacity = None
        if position.stressed_market_value is not None:
            available_capacity = (
                position.stressed_market_value * position.stressed_liquidity_capacity_rate
            )
        eligible_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "position_id": position.position_id,
                "asset_group": position.asset_group.value,
                "stressed_market_value": money(position.stressed_market_value),
                "stressed_haircut_rate": rate(position.stressed_haircut_rate),
                "stressed_liquidity_capacity_rate": rate(position.stressed_liquidity_capacity_rate),
                "available_capacity": money(available_capacity),
                "settlement_days": days(position.settlement_days),
                "maturity_days": days(position.maturity_days),
                "stress_horizon_days": horizon,
                "eligible_for_liquidation": is_eligible(position, horizon),
            }
        )

pd.DataFrame(eligible_rows)

,scenario_id,position_id,asset_group,stressed_market_value,stressed_haircut_rate,stressed_liquidity_capacity_rate,available_capacity,settlement_days,maturity_days,stress_horizon_days,eligible_for_liquidation
0,moderate_redemption_most_liquid,eur_operating_cash,cash,"12,000,000.00",0.00%,100.00%,"12,000,000.00",0,,5,False
1,moderate_redemption_most_liquid,sap_equity_position,listed_equity,"15,000,000.00",0.00%,20.00%,"3,000,000.00",2,,5,True
2,moderate_redemption_most_liquid,asml_equity_position,listed_equity,"12,000,000.00",0.00%,18.00%,"2,160,000.00",2,,5,True
3,moderate_redemption_most_liquid,lvmh_equity_position,listed_equity,"8,000,000.00",0.00%,18.00%,"1,440,000.00",2,,5,True
4,moderate_redemption_most_liquid,msci_world_etf_position,listed_etf,"18,000,000.00",0.00%,35.00%,"6,300,000.00",2,,5,True
5,moderate_redemption_most_liquid,euro_stoxx_etf_position,listed_etf,"10,000,000.00",0.00%,40.00%,"4,000,000.00",2,,5,True
6,moderate_redemption_most_liquid,overnight_reverse_repo_bnp,reverse_repo,"15,000,000.00",0.00%,100.00%,"15,000,000.00",1,1,5,True
7,moderate_redemption_most_liquid,one_week_reverse_repo_sg,reverse_repo,"10,000,000.00",0.00%,90.00%,"9,000,000.00",1,7,5,False
8,moderate_redemption_most_liquid,eur_repo_financing_obligation,repo_financing,,0.00%,0.00%,,1,,5,False
9,platform_outflow_pro_rata,eur_operating_cash,cash,"12,000,000.00",0.00%,100.00%,"12,000,000.00",0,,5,False


## Liquidation Strategy Execution

This section summarizes the engine output for each scenario after calling `calculate_liquidation_strategy`. It matters because this is the central review point: the engine must convert redemption need and stressed liquidity assumptions into cash raised, dilution, shortfall, and remaining buffer metrics.

Interpret `shortfall` as unmet redemption need after cash and post-haircut asset sales. Interpret `remaining_liquid_buffer_rate` as post-haircut eligible liquidity plus remaining cash divided by NAV.

In [10]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "total_redemption_amount": money(run["result"].total_redemption_amount),
            "cash_used": money(run["result"].cash_used),
            "post_haircut_cash_raised": money(run["result"].total_post_haircut_cash_raised),
            "dilution_amount": money(run["result"].dilution_amount),
            "dilution_rate": rate(run["result"].dilution_rate),
            "shortfall": money(run["result"].shortfall),
            "remaining_liquid_buffer_rate": rate(run["result"].remaining_liquid_buffer_rate),
            "minimum_cash_buffer_preserved": run["result"].minimum_cash_buffer_preserved,
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,total_redemption_amount,cash_used,post_haircut_cash_raised,dilution_amount,dilution_rate,shortfall,remaining_liquid_buffer_rate,minimum_cash_buffer_preserved
0,moderate_redemption_most_liquid,most_liquid_first,"11,500,000.00","7,000,000.00","4,500,000.00",0.00,0.00%,0.00,32.40%,True
1,platform_outflow_pro_rata,pro_rata,"17,250,000.00",0.00,"10,989,307.69","845,700.16",0.85%,"6,260,692.31",16.03%,True
2,platform_outflow_hybrid,hybrid,"17,250,000.00","3,500,000.00","10,239,051.28","825,252.63",0.83%,"3,510,948.72",13.28%,True
3,extreme_outflow_custom_weights,custom_weights,"23,000,000.00",0.00,"9,455,333.33","920,268.04",0.92%,"13,544,666.67",12.25%,True


## Liquidation Allocation

This section shows the position-level gross sale allocation selected by each strategy. It matters because the strategy choice should be visible in which assets are used and in what amount.

Interpret `gross_sale_amount` as the amount sold before haircut. For positive haircut scenarios, gross sales can be higher than the post-haircut cash raised.

In [11]:
allocation_rows = []
for run in runs:
    for asset in run["result"].assets_liquidated:
        allocation_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "strategy_type": run["strategy"].strategy_type.value,
                "position_id": asset.position_id,
                "asset_group": asset.asset_group.value,
                "gross_sale_amount": money(asset.gross_sale_amount),
            }
        )

pd.DataFrame(allocation_rows)

,scenario_id,strategy_type,position_id,asset_group,gross_sale_amount
0,moderate_redemption_most_liquid,most_liquid_first,overnight_reverse_repo_bnp,reverse_repo,"4,500,000.00"
1,platform_outflow_pro_rata,pro_rata,asml_equity_position,listed_equity,"1,080,000.00"
2,platform_outflow_pro_rata,pro_rata,euro_stoxx_etf_position,listed_etf,"2,000,000.00"
3,platform_outflow_pro_rata,pro_rata,lvmh_equity_position,listed_equity,"720,000.00"
4,platform_outflow_pro_rata,pro_rata,msci_world_etf_position,listed_etf,"3,150,000.00"
5,platform_outflow_pro_rata,pro_rata,overnight_reverse_repo_bnp,reverse_repo,"3,385,007.85"
6,platform_outflow_pro_rata,pro_rata,sap_equity_position,listed_equity,"1,500,000.00"
7,platform_outflow_hybrid,hybrid,asml_equity_position,listed_equity,"1,080,000.00"
8,platform_outflow_hybrid,hybrid,euro_stoxx_etf_position,listed_etf,"1,916,109.25"
9,platform_outflow_hybrid,hybrid,lvmh_equity_position,listed_equity,"720,000.00"


## Post-Haircut Cash Raised

This section shows the cash generated after applying scenario-driven haircut rates to gross sales. It matters because the engine targets post-haircut cash need when capacity allows, rather than treating gross sale amount as usable cash.

Interpret the difference between `gross_sale_amount` and `post_haircut_cash_raised` as haircut leakage that contributes to dilution cost.

In [12]:
post_haircut_rows = []
for run in runs:
    for asset in run["result"].assets_liquidated:
        post_haircut_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "position_id": asset.position_id,
                "asset_group": asset.asset_group.value,
                "gross_sale_amount": money(asset.gross_sale_amount),
                "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
                "haircut_cost": money(asset.haircut_cost),
            }
        )

pd.DataFrame(post_haircut_rows)

,scenario_id,position_id,asset_group,gross_sale_amount,post_haircut_cash_raised,haircut_cost
0,moderate_redemption_most_liquid,overnight_reverse_repo_bnp,reverse_repo,"4,500,000.00","4,500,000.00",0.00
1,platform_outflow_pro_rata,asml_equity_position,listed_equity,"1,080,000.00","950,400.00","129,600.00"
2,platform_outflow_pro_rata,euro_stoxx_etf_position,listed_etf,"2,000,000.00","1,840,000.00","160,000.00"
3,platform_outflow_pro_rata,lvmh_equity_position,listed_equity,"720,000.00","633,600.00","86,400.00"
4,platform_outflow_pro_rata,msci_world_etf_position,listed_etf,"3,150,000.00","2,898,000.00","252,000.00"
5,platform_outflow_pro_rata,overnight_reverse_repo_bnp,reverse_repo,"3,385,007.85","3,317,307.69","67,700.16"
6,platform_outflow_pro_rata,sap_equity_position,listed_equity,"1,500,000.00","1,350,000.00","150,000.00"
7,platform_outflow_hybrid,asml_equity_position,listed_equity,"1,080,000.00","950,400.00","129,600.00"
8,platform_outflow_hybrid,euro_stoxx_etf_position,listed_etf,"1,916,109.25","1,762,820.51","153,288.74"
9,platform_outflow_hybrid,lvmh_equity_position,listed_equity,"720,000.00","633,600.00","86,400.00"


## Dilution Cost

This section aggregates haircut cost into the engine's dilution estimate. It matters because dilution feeds later LMT warning checks, especially swing-pricing threshold analysis.

Interpret higher `dilution_rate` as more liquidation cost relative to NAV. The inspection does not decide whether an LMT should be activated.

In [13]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "dilution_amount": money(run["result"].dilution_amount),
            "dilution_rate": rate(run["result"].dilution_rate),
            "swing_threshold_rate": rate(run["parameters"].swing_threshold_rate),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,dilution_amount,dilution_rate,swing_threshold_rate
0,moderate_redemption_most_liquid,most_liquid_first,0.00,0.00%,1.50%
1,platform_outflow_pro_rata,pro_rata,"845,700.16",0.85%,1.50%
2,platform_outflow_hybrid,hybrid,"825,252.63",0.83%,1.50%
3,extreme_outflow_custom_weights,custom_weights,"920,268.04",0.92%,1.00%


## Shortfall

This section isolates remaining unmet redemption need after cash use and post-haircut liquidation proceeds. It matters because shortfall indicates that eligible liquidity, after capacity and haircut constraints, was insufficient for the scenario.

Interpret zero shortfall as the strategy raising enough cash within the one-period stress horizon. Positive shortfall should be reviewed as a methodology or parameter concern in a separate ticket, not changed inside this inspection.

In [14]:
pd.DataFrame(
    [
        {
            "scenario_id": run["scenario"].scenario_id,
            "strategy_type": run["strategy"].strategy_type.value,
            "total_redemption_amount": money(run["result"].total_redemption_amount),
            "total_cash_raised": money(
                run["result"].cash_used + run["result"].total_post_haircut_cash_raised
            ),
            "shortfall": money(run["result"].shortfall),
        }
        for run in runs
    ]
)

,scenario_id,strategy_type,total_redemption_amount,total_cash_raised,shortfall
0,moderate_redemption_most_liquid,most_liquid_first,"11,500,000.00","11,500,000.00",0.00
1,platform_outflow_pro_rata,pro_rata,"17,250,000.00","10,989,307.69","6,260,692.31"
2,platform_outflow_hybrid,hybrid,"17,250,000.00","13,739,051.28","3,510,948.72"
3,extreme_outflow_custom_weights,custom_weights,"23,000,000.00","9,455,333.33","13,544,666.67"


## Allocation By Asset Group

This section groups gross liquidation allocations by asset group. It matters because reviewers can compare strategy behavior at a portfolio level without reading every position row.

Interpret these values as gross allocation amounts, not post-haircut cash raised. Use the post-haircut section to inspect usable cash.

In [15]:
asset_group_rows = []
for run in runs:
    for asset_group, allocation in run["result"].asset_group_allocations.items():
        asset_group_rows.append(
            {
                "scenario_id": run["scenario"].scenario_id,
                "strategy_type": run["strategy"].strategy_type.value,
                "asset_group": asset_group.value,
                "gross_allocation_amount": money(allocation),
            }
        )

pd.DataFrame(asset_group_rows).sort_values(["scenario_id", "asset_group"])

,scenario_id,strategy_type,asset_group,gross_allocation_amount
11,extreme_outflow_custom_weights,custom_weights,listed_equity,"2,200,000.00"
10,extreme_outflow_custom_weights,custom_weights,listed_etf,"3,433,333.33"
9,extreme_outflow_custom_weights,custom_weights,reverse_repo,"4,742,268.04"
0,moderate_redemption_most_liquid,most_liquid_first,cash,"7,000,000.00"
1,moderate_redemption_most_liquid,most_liquid_first,reverse_repo,"4,500,000.00"
5,platform_outflow_hybrid,hybrid,cash,"3,500,000.00"
6,platform_outflow_hybrid,hybrid,listed_equity,"3,300,000.00"
7,platform_outflow_hybrid,hybrid,listed_etf,"5,066,109.25"
8,platform_outflow_hybrid,hybrid,reverse_repo,"2,698,194.66"
2,platform_outflow_pro_rata,pro_rata,listed_equity,"3,300,000.00"


## Reviewer Notes

Use this inspection as a review aid. If the tables reveal a methodology gap, unexpected edge case, or desired refinement, capture it as a separate implementation ticket.

Out of scope for this inspection: new liquidation strategy types, multi-period liquidation, Streamlit UI, production reporting, performance optimization, and changes to the core methodology or package APIs.